# SWC Skeleton Exploration
This notebook loads a sample SWC neuron/dendrite skeleton file (`sample.swc`), parses it into a tabular form, builds a tree graph, and computes basic morphometric and geometric features including a principal (SO(2)) axis and cylindrical coordinates.

## SWC Format Refresher
Standard SWC columns: id, type, x, y, z, radius, parent.
Parent of 0 (or -1 in some variants) indicates the root. Types are integer codes (e.g. 1=soma, 3=dendrite).
Our sample file uses commas as delimiters; many SWC files use spaces. We'll implement a flexible parser.

In [ ]:
import pandas as pd, numpy as np, networkx as nx, math, pathlib, matplotlib.pyplot as plt
from collections import deque
from typing import Tuple
DATA_PATH = pathlib.Path('/Volumes/Seagate/itet-stor/guptau/net_scratch/small_trees/val/SEW24_GD_161_cor.csv.swc')
assert DATA_PATH.exists(), f'Missing sample SWC: {DATA_PATH}'
raw_lines = [ln.strip() for ln in DATA_PATH.read_text().splitlines() if ln.strip() and not ln.startswith('#')]
raw_lines[:5]

['1 3 30.264878 -15.605324 -1.012191 1.000000 0',
 '2 3 30.242273 -15.583112 -0.115535 1.000000 1',
 '3 3 30.211908 -15.495388 0.252111 1.000000 2',
 '4 3 30.223588 -15.565191 -0.124082 1.000000 2',
 '5 3 30.191461 -15.589761 0.885491 1.000000 3']

In [134]:
def parse_swc_lines(lines):
    records = []
    for ln in lines:
        # Support comma or whitespace separation
        if ',' in ln: parts = ln.split(',')
        else: parts = ln.split()
        if len(parts) < 7: raise ValueError(f'Bad SWC line: {ln}')
        id_, typ, x, y, z, rad, parent = parts[:7]
        records.append((int(id_), int(typ), float(x), float(y), float(z), float(rad), int(parent)))
    df = pd.DataFrame(records, columns=['id','type','x','y','z','radius','parent'])
    return df.sort_values('id').reset_index(drop=True)
df = parse_swc_lines(raw_lines)
df.head(10)

,id,type,x,y,z,radius,parent
0,1,3,30.264878,-15.605324,-1.012191,1.0,0
1,2,3,30.242273,-15.583112,-0.115535,1.0,1
2,3,3,30.211908,-15.495388,0.252111,1.0,2
3,4,3,30.223588,-15.565191,-0.124082,1.0,2
4,5,3,30.191461,-15.589761,0.885491,1.0,3
5,6,3,30.154361,-15.369348,0.154610,1.0,3
6,7,3,30.057500,-15.487007,-0.172594,1.0,4
7,8,3,30.139213,-15.400591,-0.190997,1.0,4
8,9,3,30.190253,-15.591650,1.024831,1.0,5
9,10,3,29.937166,-15.574710,0.885362,1.0,5


## Basic Integrity Checks

In [135]:
def swc_integrity(df: pd.DataFrame):
    # Parent must reference an existing id or be 0 (root).
    ids = set(df.id)
    bad = [p for p in df.parent if p not in ids and p != 0]
    root_candidates = df[df.parent==0].id.tolist()
    return {'num_nodes': len(df), 'bad_parent_refs': bad, 'num_root_children': len(root_candidates), 'root_children': root_candidates}
swc_integrity(df)

{'num_nodes': 104,
 'bad_parent_refs': [],
 'num_root_children': 1,
 'root_children': [1]}

## Helper Methods

(1) Take care of high-degree (>3) branching nodes. 4 deg nodes split into two branches + > 4 deg only branches with longest radii retained (thickest branches)
(2) Remove intermediary deg 2 nodes that are only there to capture shape
(3) Change simplified skeleton to networkx graph

In [136]:
def preprocess_high_degree_nodes(
    df: pd.DataFrame,
    *,
    root_parent_value: int = 0,
    eps_scale: float = 0.1,
    new_node_type: int | None = None,   # default: inherit from split node if None
    keep_two_by: str = "radius"         # for deg>4: 'radius' at child node
) -> pd.DataFrame:
    """
    Normalize high-degree branchpoints in an SWC:
      - If a non-root node has degree==4 (1 parent + 3 children), split into two degree-3 nodes by
        inserting a new synthetic node slightly offset towards the bisector of the two 'closest' branches
        (closest to each other; equivalently, not the farthest-by-angle from the parent).
      - If degree>4, keep only the two children with largest 'keep_two_by' (default child radius) and
        drop the other child subtrees.

    This runs BEFORE any degree-2 collapse.

    Returns a new DataFrame with the same schema and possibly additional rows for inserted nodes.
    """
    required = {'id','type','x','y','z','radius','parent'}
    if not required.issubset(df.columns):
        raise ValueError(f"SWC DataFrame missing columns: {required - set(df.columns)}")

    # Work on dictionaries for cheap local updates
    df = df.copy()
    df['id'] = df['id'].astype(int)
    df['parent'] = df['parent'].astype(int)

    # Node records
    nodes = {int(r.id): {
        'type': int(r.type),
        'x': float(r.x), 'y': float(r.y), 'z': float(r.z),
        'radius': float(r.radius),
        'parent': int(r.parent)
    } for r in df.itertuples(index=False)}

    # Children map
    children = {}
    for nid, rec in nodes.items():
        p = rec['parent']
        if p > root_parent_value:
            children.setdefault(p, []).append(nid)
        else:
            children.setdefault(p, [])  # ensure key for root_parent_value (optional)

    def get_children(u):
        return children.get(u, [])

    def get_coord(u):
        r = nodes[u]
        return np.array([r['x'], r['y'], r['z']], dtype=float)

    def norm(v):
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

    # Compute degree quickly (parent presence + number of children)
    def degree(u):
        p = nodes[u]['parent']
        return len(get_children(u)) + (1 if p > root_parent_value else 0)

    # Collect candidate nodes: "one parent & >=3 children"
    candidates = []
    for u in list(nodes.keys()):
        if u not in nodes:  # may be deleted later
            continue
        if nodes[u]['parent'] <= root_parent_value:
            continue  # root: skip by spec
        if len(get_children(u)) >= 3:
            candidates.append(u)

    # Helper: delete a subtree rooted at v (inclusive)
    def delete_subtree(v):
        stack = [v]
        to_del = []
        while stack:
            w = stack.pop()
            to_del.append(w)
            stack.extend(get_children(w))
        # Remove from children map and nodes
        for w in to_del:
            # detach from its parent list
            pw = nodes[w]['parent']
            if pw in children:
                if w in children[pw]:
                    children[pw].remove(w)
            # remove w's children bucket
            if w in children:
                del children[w]
            # remove node
            if w in nodes:
                del nodes[w]

    # First, handle deg>4 (rare, prune subtrees)
    for u in list(candidates):
        if u not in nodes:
            continue
        kids = get_children(u)
        if len(kids) <= 3:
            continue  # not >4
        # Rank children by criterion (default: child radius at the child node)
        if keep_two_by == "radius":
            key_fn = lambda c: nodes[c]['radius']
        else:
            key_fn = lambda c: nodes[c]['radius']  # extendable

        kids_sorted = sorted(kids, key=key_fn, reverse=True)
        keep = set(kids_sorted[:2])
        drop = [c for c in kids if c not in keep]

        # Delete dropped subtrees
        for c in drop:
            delete_subtree(c)

    # Recompute candidates after pruning (some nodes may now have 2 kids only)
    candidates = []
    for u in list(nodes.keys()):
        if nodes[u]['parent'] <= root_parent_value:
            continue
        if len(get_children(u)) == 3:
            candidates.append(u)

    # Now handle exactly 3 children (deg==4): split into two deg-3 by inserting u'
    max_id = max(nodes.keys()) if nodes else 0
    for u in candidates:
        # Validate current children set
        kids = get_children(u)
        if len(kids) != 3:
            continue

        p = nodes[u]['parent']
        if p <= root_parent_value:
            continue  # by spec: only nodes with a parent

        xu = get_coord(u); xp = get_coord(p)
        vp = norm(xp - xu)

        # Child directions and angles to parent
        v_list = []
        for c in kids:
            vc = norm(get_coord(c) - xu)
            # numerical guard for dot
            dot = float(np.clip(np.dot(vp, vc), -1.0, 1.0))
            theta = float(np.arccos(dot))
            v_list.append((c, vc, theta))

        # Identify farthest-by-angle child id
        c_far, v_far, _ = max(v_list, key=lambda t: t[2])
        # Collect close pair records (exclude the far child)
        close_records = [(cid, vec, ang) for (cid, vec, ang) in v_list if cid != c_far]

        if len(close_records) != 2:
            continue  # safety
        (c1, v1, _), (c2, v2, _) = close_records

        # Local scale for ε
        dists = [np.linalg.norm(get_coord(ci) - xu) for ci, _, _ in v_list]
        local_scale = np.mean(dists) if dists else 1.0
        eps = max(1e-6, eps_scale * local_scale)

        # Bisector direction (fallback to v1 if near-zero)
        bis = v1 + v2
        if np.linalg.norm(bis) < 1e-9:
            bis = v1
        bis = norm(bis)

        # Insert new node u'
        max_id += 1
        up = max_id
        nodes[up] = {
            'type': nodes[u]['type'] if new_node_type is None else int(new_node_type),
            'x': float(xu[0] + eps * bis[0]),
            'y': float(xu[1] + eps * bis[1]),
            'z': float(xu[2] + eps * bis[2]),
            'radius': float(np.median([nodes[c1]['radius'], nodes[c2]['radius'], nodes[u]['radius']])),
            'parent': u
        }

        # Update children map: u' is child of u
        children.setdefault(u, []).append(up)
        children.setdefault(up, [])

        # Reassign the two close children to u'
        for c in (c1, c2):
            if c in children[u]:
                children[u].remove(c)
            children[up].append(c)
            nodes[c]['parent'] = up
        # c_far remains child of u

    # Rebuild DataFrame from nodes dictionary
    rows = []
    for nid, rec in nodes.items():
        rows.append({
            'id': int(nid),
            'type': int(rec['type']),
            'x': float(rec['x']),
            'y': float(rec['y']),
            'z': float(rec['z']),
            'radius': float(rec['radius']),
            'parent': int(rec['parent'])
        })
    out = pd.DataFrame(rows, columns=['id','type','x','y','z','radius','parent'])

    # Keep stable id ordering
    out = out.sort_values('id').reset_index(drop=True)
    return out

In [137]:
def _euclid_len(coords):
    # coords: (k,3) array
    diffs = np.diff(coords, axis=0)
    return np.sqrt((diffs**2).sum(axis=1)).sum()


def simplify_swc(df: pd.DataFrame, *, root_parent_value=0, keep_parent_value=0):
    """
    Collapse degree-2 internal nodes in a SWC tree to keep only:
      - roots (parent <= root_parent_value)
      - branching nodes (original degree != 2)
      - terminal (leaf) nodes (original degree == 1)

    Parameters
    ----------
    df : DataFrame
        Columns: id, type, x, y, z, radius, parent (id and parent are ints)
    root_parent_value : int
        Values <= this are treated as roots. (Typical SWC uses 0 or -1.)
    keep_parent_value : int
        Parent value to assign to roots in the simplified SWC (0 or -1).

    Returns
    -------
    nodes_simplified : DataFrame
        Columns: id, orig_id, type, x, y, z, radius, parent
        (ids are re-numbered consecutively; parent uses these new ids)
    edges_simplified : DataFrame
        Columns: u_id, v_id, u_orig, v_orig, length, path_ids
        (u_id/v_id are new ids; u_orig/v_orig are original ids; path_ids is a Python list)
    """
    required = {'id','type','x','y','z','radius','parent'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"SWC DataFrame missing columns: {missing}")

    # Ensure integer ids and parents
    df = df.copy()
    df['id'] = df['id'].astype(int)
    df['parent'] = df['parent'].astype(int)

    # Build undirected graph G from parent relations
    G = nx.Graph()
    for _, r in df.iterrows():
        G.add_node(int(r['id']),
                   type=int(r['type']),
                   x=float(r['x']), y=float(r['y']), z=float(r['z']),
                   radius=float(r['radius']))
    for _, r in df.iterrows():
        pid = int(r['parent'])
        if pid > root_parent_value:
            G.add_edge(int(r['id']), pid)

    if G.number_of_nodes() == 0:
        raise ValueError("Empty graph after parsing SWC.")

    roots = df.loc[df['parent'] <= root_parent_value, 'id'].astype(int).tolist()
    if not roots:
        roots = [int(df['id'].min())]

    deg = dict(G.degree())

    # Keepers: all roots + nodes with degree != 2
    keepers = set(roots) | {n for n, d in deg.items() if d != 2}

    coords = df.set_index('id')[['x','y','z']].to_dict('index')

    def walk_to_keeper(u, v):
        path = [u, v]
        prev, cur = u, v
        while cur not in keepers:
            nbrs = list(G.neighbors(cur))
            nxt = nbrs[0] if nbrs[1] == prev else nbrs[1]
            prev, cur = cur, nxt
            path.append(cur)
        return cur, path

    seen_edges = set()
    simp_edges = []
    for u in keepers:
        for v in G.neighbors(u):
            a, b = sorted((u, v))
            if (a, b) in seen_edges:
                continue
            seen_edges.add((a, b))
            if v in keepers:
                path_ids = [u, v]
                c = np.array([[coords[u]['x'], coords[u]['y'], coords[u]['z']],
                              [coords[v]['x'], coords[v]['y'], coords[v]['z']]])
                length = _euclid_len(c)
                uu, ww = u, v
            else:
                w, path_ids = walk_to_keeper(u, v)
                c = np.array([[coords[i]['x'], coords[i]['y'], coords[i]['z']] for i in path_ids])
                length = _euclid_len(c)
                uu, ww = u, w
            k0, k1 = sorted((uu, ww))
            if (k0, k1) in {(e['u'], e['v']) for e in simp_edges}:
                continue
            simp_edges.append({
                'u': k0,
                'v': k1,
                'path_ids': path_ids if (uu, ww) == (k0, k1) else list(reversed(path_ids)),
                'length': float(length),
            })

    H = nx.Graph()
    H.add_nodes_from((k, G.nodes[k]) for k in keepers)
    for e in simp_edges:
        H.add_edge(e['u'], e['v'])

    new_id = {}
    parent_of = {}
    order = []
    visited = set()
    from collections import deque as _deque
    for r in roots:
        if r not in H or r in visited:
            continue
        q = _deque([r])
        parent_of[r] = keep_parent_value
        visited.add(r)
        while q:
            u = q.popleft()
            order.append(u)
            for v in H.neighbors(u):
                if v not in visited:
                    parent_of[v] = u
                    visited.add(v)
                    q.append(v)
    for comp in nx.connected_components(H):
        any_node = next(iter(comp))
        if any_node in visited:
            continue
        q = _deque([any_node])
        parent_of[any_node] = keep_parent_value
        visited.add(any_node)
        while q:
            u = q.popleft()
            order.append(u)
            for v in H.neighbors(u):
                if v not in visited:
                    parent_of[v] = u
                    visited.add(v)
                    q.append(v)

    for i, oid in enumerate(order, start=1):
        new_id[oid] = i

    rows = []
    for oid in order:
        nd = G.nodes[oid]
        rows.append({
            'id': new_id[oid],
            'orig_id': oid,
            'type': nd['type'],
            'x': nd['x'],
            'y': nd['y'],
            'z': nd['z'],
            'radius': nd['radius'],
            'parent': keep_parent_value if parent_of[oid] == keep_parent_value else new_id[parent_of[oid]],
        })
    nodes_simplified = pd.DataFrame(rows)[['id','orig_id','type','x','y','z','radius','parent']]

    e_rows = []
    for e in simp_edges:
        u_new = new_id[e['u']]
        v_new = new_id[e['v']]
        e_rows.append({
            'u_id': min(u_new, v_new),
            'v_id': max(u_new, v_new),
            'u_orig': min(e['u'], e['v']),
            'v_orig': max(e['u'], e['v']),
            'length': e['length'],
            'path_ids': e['path_ids'],
        })
    edges_simplified = pd.DataFrame(e_rows).drop_duplicates(subset=['u_id','v_id']).reset_index(drop=True)

    # Sanity: internal degree-2 should be absent. Allow root to have any degree.
    Hd_full = nx.Graph([(r['u_orig'], r['v_orig']) for _, r in edges_simplified.iterrows()])
    Hd = dict(Hd_full.degree())
    internal_nodes = set(Hd_full.nodes()) - set(roots)
    assert all(Hd[n] != 2 for n in internal_nodes), "Collapsed graph still has internal degree-2 nodes – check input."\
        + f" Roots (allowed) with degree-2: {[n for n in roots if Hd.get(n)==2]}"

    return nodes_simplified, edges_simplified

In [138]:
def simplified_df_to_graph(nodes_df: pd.DataFrame, edges_df: pd.DataFrame) -> nx.Graph:
    """Construct an undirected keeper graph from simplified SWC tables.

    Nodes: use `orig_id` as the graph node key to retain traceability to original SWC.
    Node attributes: x,y,z,radius,type plus new_id for reference.
    Edges: from `edges_df` using `u_orig`, `v_orig`.
    Length attribute copied from simplified edge metadata.
    Path_ids retained on edge for downstream traversal/reconstruction.
    """
    Gk = nx.Graph()
    for _, r in nodes_df.iterrows():
        Gk.add_node(int(r['orig_id']),
                    new_id=int(r['id']),
                    type=int(r['type']),
                    x=float(r['x']), y=float(r['y']), z=float(r['z']),
                    radius=float(r['radius']))
    for _, e in edges_df.iterrows():
        Gk.add_edge(int(e['u_orig']), int(e['v_orig']), length=float(e['length']), path_ids=list(e['path_ids']))
    return Gk

## Build Cleaned Graphs

In [139]:
# 0) degree-2 collapse for original
nodes_original, edges_original = simplify_swc(
    df,
    root_parent_value=0,
    keep_parent_value=0
)

# 1) Normalize high-degree branchpoints on the raw SWC
df_norm = preprocess_high_degree_nodes(
    df,
    root_parent_value=0,
    eps_scale=0.1,        # tweak if needed
    new_node_type=None,   # inherit 'type' from source node; or set 3 explicitly
    keep_two_by="radius"  # for deg>4 pruning
)


# 2) Then run your (unchanged) degree-2 collapse
nodes_simplified, edges_simplified = simplify_swc(
    df_norm,
    root_parent_value=0,
    keep_parent_value=0
)

G_original = simplified_df_to_graph(nodes_original, edges_original)
G_simplified = simplified_df_to_graph(nodes_simplified, edges_simplified)


## 3D Visualizations

In [140]:
import plotly.graph_objects as go

# Build coordinate and color arrays
xs=[]; ys=[]; zs=[]; colors=[]; sizes=[]
for n, data in G_original.nodes(data=True):
    x,y,z = data['x'], data['y'], data['z']
    xs.append(x); ys.append(y); zs.append(z)
    # color soma/type==1 differently
    colors.append('darkorange' if data.get('type') == 1 else 'dodgerblue')
    d = G_original.degree[n]
    if d == 1: sizes.append(3)
    elif d == 2: sizes.append(5)
    else: sizes.append(8)

# Create edge segments as separate scatter traces for better hover clarity
edge_x=[]; edge_y=[]; edge_z=[]; edge_i=[]
for i,(u,v) in enumerate(G_original.edges()):
    pu = (G_original.nodes[u]['x'], G_original.nodes[u]['y'], G_original.nodes[u]['z'])
    pv = (G_original.nodes[v]['x'], G_original.nodes[v]['y'], G_original.nodes[v]['z'])
    edge_x.extend([pu[0], pv[0], None])
    edge_y.extend([pu[1], pv[1], None])
    edge_z.extend([pu[2], pv[2], None])

node_trace = go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='markers',
    marker=dict(size=sizes, color=colors, opacity=0.9),
    text=[f"id={n} deg={G_original.degree[n]} type={G_original.nodes[n]['type']}" for n in G_original.nodes()],
    hoverinfo='text'
)

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='gray', width=2),
    hoverinfo='skip'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title='SWC Skeleton (Interactive)',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

In [141]:
import plotly.graph_objects as go

# Build coordinate and color arrays
xs=[]; ys=[]; zs=[]; colors=[]; sizes=[]
for n, data in G_simplified.nodes(data=True):
    x,y,z = data['x'], data['y'], data['z']
    xs.append(x); ys.append(y); zs.append(z)
    # color soma/type==1 differently
    colors.append('darkorange' if data.get('type') == 1 else 'dodgerblue')
    d = G_simplified.degree[n]
    if d == 1: sizes.append(3)
    elif d == 2: sizes.append(5)
    else: sizes.append(8)

# Create edge segments as separate scatter traces for better hover clarity
edge_x=[]; edge_y=[]; edge_z=[]; edge_i=[]
for i,(u,v) in enumerate(G_simplified.edges()):
    pu = (G_simplified.nodes[u]['x'], G_simplified.nodes[u]['y'], G_simplified.nodes[u]['z'])
    pv = (G_simplified.nodes[v]['x'], G_simplified.nodes[v]['y'], G_simplified.nodes[v]['z'])
    edge_x.extend([pu[0], pv[0], None])
    edge_y.extend([pu[1], pv[1], None])
    edge_z.extend([pu[2], pv[2], None])

node_trace = go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='markers',
    marker=dict(size=sizes, color=colors, opacity=0.9),
    text=[f"id={n} deg={G_simplified.degree[n]} type={G_simplified.nodes[n]['type']}" for n in G_simplified.nodes()],
    hoverinfo='text'
)

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='gray', width=2),
    hoverinfo='skip'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title='SWC Skeleton (Interactive)',
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

In [ ]:
import plotly.graph_objects as go
try:
    from ipywidgets import interact, IntSlider
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False

import networkx as nx
from collections import deque

def depth_limited_subgraph(G: nx.Graph, root: int, max_depth: int) -> nx.Graph:
    """Return an induced subgraph containing nodes up to max_depth (BFS levels from root)."""
    depths = {root: 0}
    q = deque([root])
    keep = {root}
    while q:
        u = q.popleft()
        if depths[u] == max_depth:
            continue
        for v in G.neighbors(u):
            if v not in depths:
                depths[v] = depths[u] + 1
                if depths[v] <= max_depth:
                    keep.add(v)
                q.append(v)
    H = G.subgraph(keep).copy()
    return H

def choose_root(G: nx.Graph) -> int:
    """Heuristic root: prefer a soma node (type==1) else smallest id."""
    soma = [n for n, d in G.nodes(data=True) if d.get('type') == 1]
    if soma:
        return soma[0]
    return min(G.nodes())

root_candidate = choose_root(G_original)

def make_depth_figure(depth: int):
    H = depth_limited_subgraph(G_original, root_candidate, depth)
    xs=[]; ys=[]; zs=[]; colors=[]; sizes=[]
    for n,data in H.nodes(data=True):
        xs.append(data['x']); ys.append(data['y']); zs.append(data['z'])
        colors.append('darkorange' if data.get('type')==1 else 'dodgerblue')
        d = H.degree[n]
        sizes.append(3 if d==1 else (5 if d==2 else 8))
    edge_x=[]; edge_y=[]; edge_z=[]
    for (u,v) in H.edges():
        pu = (H.nodes[u]['x'], H.nodes[u]['y'], H.nodes[u]['z'])
        pv = (H.nodes[v]['x'], H.nodes[v]['y'], H.nodes[v]['z'])
        edge_x.extend([pu[0], pv[0], None])
        edge_y.extend([pu[1], pv[1], None])
        edge_z.extend([pu[2], pv[2], None])

    node_trace = go.Scatter3d(x=xs,y=ys,z=zs, mode='markers',
                              marker=dict(size=sizes, color=colors, opacity=0.9),
                              text=[f"id={n} deg={H.degree[n]}" for n in H.nodes()],
                              hoverinfo='text')
    edge_trace = go.Scatter3d(x=edge_x,y=edge_y,z=edge_z, mode='lines',
                              line=dict(color='gray', width=2), hoverinfo='skip')
    fig = go.Figure(data=[edge_trace,node_trace])
    fig.update_layout(title=f'Depth <= {depth} (|V|={H.number_of_nodes()})',
                      scene=dict(aspectmode='data'), margin=dict(l=0,r=0,b=0,t=30))
    return fig

if _HAS_WIDGETS:
    interact(lambda d: make_depth_figure(d).show(), d=IntSlider(min=1, max=100, step=1, value=20, description='Depth'))
else:
    print('ipywidgets not installed; displaying static example. Install ipywidgets for slider interactivity.')
    make_depth_figure(20).show()

interactive(children=(IntSlider(value=20, description='Depth', min=1), Output()), _dom_classes=('widget-intera…

In [143]:
import plotly.graph_objects as go
try:
    from ipywidgets import interact, IntSlider
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False

import networkx as nx
from collections import deque

def depth_limited_subgraph(G: nx.Graph, root: int, max_depth: int) -> nx.Graph:
    """Return an induced subgraph containing nodes up to max_depth (BFS levels from root)."""
    depths = {root: 0}
    q = deque([root])
    keep = {root}
    while q:
        u = q.popleft()
        if depths[u] == max_depth:
            continue
        for v in G.neighbors(u):
            if v not in depths:
                depths[v] = depths[u] + 1
                if depths[v] <= max_depth:
                    keep.add(v)
                q.append(v)
    H = G.subgraph(keep).copy()
    return H

def choose_root(G: nx.Graph) -> int:
    """Heuristic root: prefer a soma node (type==1) else smallest id."""
    soma = [n for n, d in G.nodes(data=True) if d.get('type') == 1]
    if soma:
        return soma[0]
    return min(G.nodes())

root_candidate = choose_root(G_simplified)

def make_depth_figure(depth: int):
    H = depth_limited_subgraph(G_simplified, root_candidate, depth)
    xs=[]; ys=[]; zs=[]; colors=[]; sizes=[]
    for n,data in H.nodes(data=True):
        xs.append(data['x']); ys.append(data['y']); zs.append(data['z'])
        colors.append('darkorange' if data.get('type')==1 else 'dodgerblue')
        d = H.degree[n]
        sizes.append(3 if d==1 else (5 if d==2 else 8))
    edge_x=[]; edge_y=[]; edge_z=[]
    for (u,v) in H.edges():
        pu = (H.nodes[u]['x'], H.nodes[u]['y'], H.nodes[u]['z'])
        pv = (H.nodes[v]['x'], H.nodes[v]['y'], H.nodes[v]['z'])
        edge_x.extend([pu[0], pv[0], None])
        edge_y.extend([pu[1], pv[1], None])
        edge_z.extend([pu[2], pv[2], None])

    node_trace = go.Scatter3d(x=xs,y=ys,z=zs, mode='markers',
                              marker=dict(size=sizes, color=colors, opacity=0.9),
                              text=[f"id={n} deg={H.degree[n]}" for n in H.nodes()],
                              hoverinfo='text')
    edge_trace = go.Scatter3d(x=edge_x,y=edge_y,z=edge_z, mode='lines',
                              line=dict(color='gray', width=2), hoverinfo='skip')
    fig = go.Figure(data=[edge_trace,node_trace])
    fig.update_layout(title=f'Depth <= {depth} (|V|={H.number_of_nodes()})',
                      scene=dict(aspectmode='data'), margin=dict(l=0,r=0,b=0,t=30))
    return fig

if _HAS_WIDGETS:
    interact(lambda d: make_depth_figure(d).show(), d=IntSlider(min=1, max=100, step=1, value=20, description='Depth'))
else:
    print('ipywidgets not installed; displaying static example. Install ipywidgets for slider interactivity.')
    make_depth_figure(20).show()

interactive(children=(IntSlider(value=20, description='Depth', min=1), Output()), _dom_classes=('widget-intera…

## Principal Axis (SO(2) Symmetry Candidate)

In [144]:
def principal_axis(positions: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    # Center positions
    C = positions.mean(axis=0)
    Xc = positions - C
    # PCA via SVD
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    axis = Vt[0]  # principal component
    # Build orthonormal frame: axis, v, w
    # pick any vector not collinear to axis
    cand = np.array([1,0,0],dtype=float)
    if abs(np.dot(axis, cand)) > 0.9: cand = np.array([0,1,0],dtype=float)
    v = cand - np.dot(cand, axis)*axis
    v /= np.linalg.norm(v)
    w = np.cross(axis, v)
    return axis, np.vstack([axis, v, w])
positions = np.vstack([data['pos'] for _,data in G.nodes(data=True)])
axis, frame = principal_axis(positions)
axis, frame

NameError: name 'G' is not defined